# Regime Engine v2 - Calibration Q6

**Round 6** tunes recovery-window responsiveness on top of the corrected Q5 baseline.

Q5 fixed the calibration workflow so macro-nowcast uses the `engine:` block from [`fred_series.yml`](../../configs/regime_detection/fred_series.yml). Q6 keeps that corrected baseline and applies a narrow config-only pass in [`regime_engine.yml`](../../configs/regime_detection/regime_engine.yml).

**Goal**: make benign recovery windows show through faster without pushing 2022 or 2023-24 back into persistent inflation-hot labels.

**Changes applied**:

* Growth layer blend: macro / market from `0.45 / 0.55` to `0.35 / 0.65`.
* Inflation layer blend: macro / market from `0.40 / 0.60` to `0.30 / 0.70`.
* Axis deadband: growth from `+/-0.20` to `+/-0.15`; inflation from `+/-0.20` to `+/-0.12`.
* Risk overlay thresholds were left unchanged. Q6 is a recovery-responsiveness pass, not a stress-overlay pass.


## Candidate Readout

| candidate | growth weights macro/market | inflation weights macro/market | thresholds G/I | 2017 | 2020 H2-2021 | 2022 | 2023-24 | note |
|---|---:|---:|---:|---|---|---|---|---|
| Q5 baseline | `0.45 / 0.55` | `0.40 / 0.60` | `0.20 / 0.20` | Neutral/Mixed | Neutral/Mixed | Neutral/Mixed | Neutral/Mixed | Too muted in benign recovery windows |
| A | `0.30 / 0.70` | `0.30 / 0.70` | `0.15 / 0.12` | Goldilocks | Reflation | Neutral/Mixed | Neutral/Mixed | Works, but shifts more aggressively to market layer |
| D selected | `0.35 / 0.65` | `0.30 / 0.70` | `0.15 / 0.12` | Goldilocks | Reflation | Neutral/Mixed | Neutral/Mixed | Same anchor improvement with slightly more macro anchor on growth |
| Risk variant | `0.30 / 0.70` | `0.30 / 0.70` | `0.15 / 0.12` | Goldilocks | Reflation | Neutral/Mixed | Neutral/Mixed | Raised 2018 stress but made April 2025 stress too large for this round |

**Decision**: select the moderate market-heavier blend. It corrects the most visible recovery misses without combining recovery tuning with risk-overlay tuning.


## Anchor Results After Q6

| anchor | Q5 majority | Q6 majority | stress share | disagreement share | macro G/I | market G/I |
|---|---|---|---:|---:|---:|---:|
| 2008-09 GFC | Slowdown / Deflationary Slowdown + Stress Overlay | Slowdown / Deflationary Slowdown + Stress Overlay | `0.286` | `0.682` | `-0.87 / -0.19` | `-0.02 / -0.24` |
| 2009-10 Recovery | Neutral/Mixed Growth / Neutral/Mixed Inflation | Neutral/Mixed Growth / Neutral/Mixed Inflation | `0.000` | `0.254` | `-0.15 / -0.35` | `0.16 / 0.08` |
| 2017 Soft Landing | Neutral/Mixed Growth / Neutral/Mixed Inflation | Goldilocks / Expansion | `0.000` | `0.292` | `0.18 / -0.40` | `0.19 / 0.16` |
| 2018 Q4 Selloff | Neutral/Mixed Growth / Down Inflation | Slowdown / Deflationary Slowdown | `0.091` | `0.000` | `0.13 / -0.14` | `-0.36 / -0.36` |
| 2020 COVID Shock | Slowdown / Deflationary Slowdown + Stress Overlay | Slowdown / Deflationary Slowdown + Stress Overlay | `0.440` | `0.000` | `-0.36 / -0.36` | `-0.36 / -0.51` |
| 2020 H2-2021 Reopening | Neutral/Mixed Growth / Neutral/Mixed Inflation | Reflation | `0.000` | `0.374` | `0.11 / -0.14` | `0.19 / 0.26` |
| 2022 Inflation Shock / Tightening | Neutral/Mixed Growth / Neutral/Mixed Inflation | Neutral/Mixed Growth / Neutral/Mixed Inflation | `0.000` | `0.573` | `0.31 / 0.57` | `-0.17 / -0.05` |
| 2023-24 Disinflation / Soft Landing | Neutral/Mixed Growth / Neutral/Mixed Inflation | Neutral/Mixed Growth / Neutral/Mixed Inflation | `0.000` | `0.410` | `-0.01 / 0.53` | `0.07 / -0.08` |
| 2025 April Liberation Day Tariff Shock | Neutral/Mixed Growth / Neutral/Mixed Inflation | Down Growth / Neutral/Mixed Inflation | `0.114` | `0.000` | `0.05 / 0.18` | `-0.31 / -0.11` |

The main success case is `2020 H2-2021 Reopening`: it moves from neutral to `Reflation`. `2017 Soft Landing` moves from neutral to `Goldilocks / Expansion`. `2022` and `2023-24` stay neutral-majority rather than reverting to persistent inflation-hot labels.


## 2009-10 Recovery Remains Mixed

2009-10 did not move to a majority recovery label under the selected Q6 config.

| window | final regime counts after Q6 |
|---|---|
| 2009-10 Recovery | Neutral/Mixed `136`, Down Inflation `94`, Goldilocks `54`, Up Inflation `43`, Reflation `36`, Stagflation-like `15` |

The reason is visible in the layer means: macro growth is still negative (`-0.15`) while market growth is only modestly positive (`0.16`). Making 2009 majority recovery would require lowering the growth threshold close to `0.05` or giving the market layer much more growth weight. That would overfit one anchor and make the engine more sensitive to weak market rebounds.

**Decision**: accept 2009 as a lagged macro recovery window for now. If Q7 tackles it, use a dedicated faster growth input such as claims, yield curve, or housing rather than shrinking the global deadband further.


## Side Effects To Watch

* April 2025 now shows `Down Growth / Neutral/Mixed Inflation` as the majority label. This is directionally consistent with a short-lived market dislocation, but it should be reviewed against a future risk-overlay pass.
* 2022 has a higher disagreement share (`0.573`) because macro inflation remains positive while market-implied inflation is weak. This is acceptable context under the v2 contract, but the dashboard should make the disagreement visible.
* Risk overlay remains conservative. Q6 intentionally did not lower stress thresholds because the risk variant made April 2025 stress share too high for a recovery-focused round.


## Validation

* `conda run -n py313 python -m market_helper.cli.main regime-calibrate`
* Calibration output read from `data/artifacts/regime_detection/calibration/regime_v2_calibration_summary.json`.

Next useful calibration round: separate risk-overlay tuning for GFC / 2018 Q4 / April 2025, or add a faster macro growth input before trying to force 2009-10 into a recovery majority.
